# Green NLP Benchmark Lab — Notebook 1: Baseline Measurement — coding workshop (advanced)

**Session 1 of 3.** Today you build the *baseline* version of a small pipeline that answers
questions about a handful of documents. You will measure how much energy and estimated CO2
it costs to run, and how good the answers are. Sessions 2 and 3 will change **one thing** each
time (quantization, then retrieval) and you'll compare against what you measure today.

Machine translation, and the cost of evaluating it, are in Notebook 01e.

**What you will learn:**

- to run a language model on a question-answering task, with eight documents in its input;
- to measure the energy and CO2eq of that run with CodeCarbon, and why these numbers are estimates;
- to score the output against reference answers, and what a run without the documents
  (closed-book) shows about what they contribute and what they cost;
- to save the baseline row that Sessions 2 and 3 compare against.

**Before you start:** `Runtime` → `Change runtime type` → select **T4 GPU**, then `Save`.

**At the end of this notebook** you will save a file called `leaderboard.csv`. Download it
from the Files sidebar (folder icon, left side) and keep it somewhere you can find it.

Expected time: 60–75 minutes.

## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried, or run the
   cell under it: that puts the solution into a new code cell for you, ready to run.
   Or tick **fill_in_solutions** in the cell right below these instructions and run that cell:
   from then on, every exercise cell you run gets its solution in place of its code. With the box
   ticked, Runtime → Run all runs the whole notebook with the solutions.

A cell that still contains `___` stops with `ExerciseGaps: this cell still has gaps marked ___`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

In [ ]:
# @title Run this cell first. Tick the box to fill in the solutions (point 3 above).
fill_in_solutions = False  # @param {type:"boolean"}

import io
import re
import tokenize

_solutions = {
    1: "def build_prompt(question, corpus):\n    context = \"\\n\\n\".join(f\"[{name}]\\n{text.strip()}\" for name, text in corpus.items())\n    messages = [\n        {\n            \"role\": \"system\",\n            \"content\": (\n                \"You are a concise assistant. Answer the question using ONLY the \"\n                \"documents provided below. Answer in one short sentence.\"\n            ),\n        },\n        {\n            \"role\": \"user\",\n            \"content\": f\"Documents:\\n{context}\\n\\nQuestion: {question}\",\n        },\n    ]\n    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n\ndef answer_question(question, corpus, max_new_tokens=60):\n    prompt = build_prompt(question, corpus)\n    inputs = tokenizer(prompt, return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        output = model.generate(\n            **inputs,\n            max_new_tokens=max_new_tokens,\n            do_sample=False,\n        )\n    generated = output[0][inputs[\"input_ids\"].shape[1]:]\n    return tokenizer.decode(generated, skip_special_tokens=True).strip()\n\n\nsample_answer = answer_question(QA_PAIRS[0][\"question\"], CORPUS)\nprint(\"Q:\", QA_PAIRS[0][\"question\"])\nprint(\"A:\", sample_answer)",
    2: "tracker = EmissionsTracker(\n    project_name=\"green-nlp-baseline\",\n    measure_power_secs=1,\n    save_to_file=False,\n    log_level=\"warning\",\n)\n\nresults = []\ntracker.start()\nstart_time = time.time()\n\nfor pair in QA_PAIRS:\n    answer = answer_question(pair[\"question\"], CORPUS)\n    results.append({\"question\": pair[\"question\"], \"reference\": pair[\"reference\"], \"answer\": answer})\n\nelapsed = time.time() - start_time\nemissions_kg = tracker.stop()\n\nprint(f\"Ran {len(QA_PAIRS)} questions in {elapsed:.1f} seconds\")\nprint(f\"Estimated emissions: {emissions_kg * 1000:.4f} g CO2eq\")",
    3: "import re\n\n\ndef words(text):\n    \"\"\"The words and numbers of a text, lowercased: \"1,287\" becomes 1287, \"1.5%\" becomes 1.5 and %.\"\"\"\n    text = re.sub(r\"(?<=\\d),(?=\\d{3}(?!\\d))\", \"\", text.lower())\n    return re.findall(r\"\\d+(?:\\.\\d+)?|%|[^\\W\\d_]+\", text)\n\n\ndef joined(ws):\n    \"\"\"The words as one string: letters run together, a number keeps a space on each side.\"\"\"\n    out = ws[0] if ws else \"\"\n    for prev, w in zip(ws, ws[1:]):\n        out += w if prev.isalpha() and w.isalpha() else \" \" + w\n    return out\n\n\ndef contains_reference(answer, reference):\n    \"\"\"True if some run of whole words in the answer reads the same as the reference.\"\"\"\n    a, target = words(answer), joined(words(reference))\n    return any(joined(a[i:j]) == target for i in range(len(a)) for j in range(i + 1, len(a) + 1))\n\n\ndef score_answer(answer, reference_list):\n    return any(contains_reference(answer, ref) for ref in reference_list)\n\n\nfor r in results:\n    r[\"correct\"] = score_answer(r[\"answer\"], r[\"reference\"])\n\naccuracy = sum(r[\"correct\"] for r in results) / len(results)\n\nfor r in results:\n    mark = \"✓\" if r[\"correct\"] else \"✗\"\n    print(f\"{mark} Q: {r['question']}\")\n    print(f\"   A: {r['answer']}\\n\")\n\nprint(f\"Accuracy: {accuracy:.0%} ({sum(r['correct'] for r in results)}/{len(results)})\")",
    4: "def answer_closed_book(question, max_new_tokens=60):\n    messages = [\n        {\"role\": \"system\",\n         \"content\": \"You are a concise assistant. Answer the question in one short sentence.\"},\n        {\"role\": \"user\", \"content\": f\"Question: {question}\"},\n    ]\n    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs = tokenizer(prompt, return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)\n    generated = output[0][inputs[\"input_ids\"].shape[1]:]\n    return tokenizer.decode(generated, skip_special_tokens=True).strip()\n\n\ntracker_closed = EmissionsTracker(\n    project_name=\"green-nlp-closed-book\",\n    measure_power_secs=1,\n    save_to_file=False,\n    log_level=\"warning\",\n)\n\nresults_closed = []\ntracker_closed.start()\nstart_time = time.time()\n\nfor pair in QA_PAIRS:\n    answer = answer_closed_book(pair[\"question\"])\n    results_closed.append({\"question\": pair[\"question\"], \"answer\": answer,\n                           \"correct\": score_answer(answer, pair[\"reference\"])})\n\nelapsed_closed = time.time() - start_time\nemissions_closed_kg = tracker_closed.stop()\naccuracy_closed = sum(r[\"correct\"] for r in results_closed) / len(results_closed)\n\nfor r_docs, r_closed in zip(results, results_closed):\n    print(f\"Q: {r_docs['question']}\")\n    print(f\"   with documents {'✓' if r_docs['correct'] else '✗'}  {r_docs['answer']}\")\n    print(f\"   no documents   {'✓' if r_closed['correct'] else '✗'}  {r_closed['answer']}\\n\")\n\ntokens_with_docs = len(tokenizer(build_prompt(QA_PAIRS[0][\"question\"], CORPUS))[\"input_ids\"])\nprint(f\"Accuracy with documents: {accuracy:.0%}   no documents: {accuracy_closed:.0%}\")\nprint(f\"Emissions with documents: {emissions_kg * 1000:.4f} g CO2eq   \"\n      f\"no documents: {emissions_closed_kg * 1000:.4f} g CO2eq\")\nprint(f\"Prompt length for the first question: {tokens_with_docs} tokens with documents\")",
    5: "N_PARAMS = sum(p.numel() for p in model.parameters())\n\n\ndef closed_book_prompt(question):\n    messages = [\n        {\"role\": \"system\",\n         \"content\": \"You are a concise assistant. Answer the question in one short sentence.\"},\n        {\"role\": \"user\", \"content\": f\"Question: {question}\"},\n    ]\n    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n\ndef count_tokens(text):\n    return len(tokenizer(text)[\"input_ids\"])\n\n\ndef forward_flops(n_tokens, n_params=N_PARAMS):\n    return 2 * n_params * n_tokens\n\n\ndef run_flops(results_list, make_prompt):\n    \"\"\"Estimated FLOPs of a whole run. make_prompt(question) returns the prompt text.\"\"\"\n    total = 0\n    for r in results_list:\n        n_tokens = count_tokens(make_prompt(r[\"question\"])) + count_tokens(r[\"answer\"])\n        total += forward_flops(n_tokens)\n    return total\n\nflops_docs = run_flops(results, lambda q: build_prompt(q, CORPUS))\nflops_closed = run_flops(results_closed, closed_book_prompt)\n\nprint(f\"With documents: {flops_docs:.2e} FLOPs\")\nprint(f\"Closed-book:    {flops_closed:.2e} FLOPs\")\nprint(f\"Ratio, estimated FLOPs:  {flops_docs / flops_closed:5.1f}x\")\nprint(f\"Ratio, measured time:    {elapsed / elapsed_closed:5.1f}x\")\nprint(f\"Ratio, measured CO2eq:   {emissions_kg / emissions_closed_kg:5.1f}x\")",
    6: "import copy\nfrom transformers import DynamicCache\n\nfull_example = build_prompt(\"PLACEHOLDER\", CORPUS)\nprefix_text = full_example[:full_example.index(\"Question: \")]\nprefix_ids = tokenizer(prefix_text, return_tensors=\"pt\")[\"input_ids\"].to(model.device)\n\n\ndef answer_with_prefix_cache(question, max_new_tokens=60):\n    inputs = tokenizer(build_prompt(question, CORPUS), return_tensors=\"pt\").to(model.device)\n    assert torch.equal(inputs[\"input_ids\"][0, :prefix_ids.shape[1]], prefix_ids[0])\n    with torch.no_grad():\n        output = model.generate(\n            **inputs,\n            past_key_values=copy.deepcopy(prefix_cache),\n            max_new_tokens=max_new_tokens,\n            do_sample=False,\n        )\n    generated = output[0][inputs[\"input_ids\"].shape[1]:]\n    return tokenizer.decode(generated, skip_special_tokens=True).strip()\n\n\ntracker_cached = EmissionsTracker(\n    project_name=\"green-nlp-prefix-cache\",\n    measure_power_secs=1,\n    save_to_file=False,\n    log_level=\"warning\",\n)\n\nresults_cached = []\ntracker_cached.start()\nstart_time = time.time()\n\nprefix_cache = DynamicCache()\nwith torch.no_grad():\n    model(prefix_ids, past_key_values=prefix_cache, use_cache=True)\n\nfor pair in QA_PAIRS:\n    results_cached.append({\"question\": pair[\"question\"],\n                           \"answer\": answer_with_prefix_cache(pair[\"question\"])})\n\nelapsed_cached = time.time() - start_time\nemissions_cached_kg = tracker_cached.stop()\n\nsame = sum(r[\"answer\"] == c[\"answer\"] for r, c in zip(results, results_cached))\nprint(f\"Prefix: {prefix_ids.shape[1]} tokens, computed once instead of {len(QA_PAIRS)} times\")\nprint(f\"Same answer as the baseline: {same}/{len(QA_PAIRS)}\")\nprint(f\"Baseline:       {elapsed:6.1f} s  {emissions_kg * 1000:.4f} g CO2eq\")\nprint(f\"Prefix caching: {elapsed_cached:6.1f} s  {emissions_cached_kg * 1000:.4f} g CO2eq\")",
    7: "def tracker_energy_kwh(tracker_obj):\n    try:\n        return float(tracker_obj.final_emissions_data.energy_consumed)\n    except (AttributeError, TypeError):\n        return None\n\n\nn_params_billion = round(sum(p.numel() for p in model.parameters()) / 1e9, 2)\n\nleaderboard = pd.DataFrame([\n    {\n        \"config\": \"baseline-fp16\",\n        \"model\": MODEL_NAME,\n        \"params_billion\": n_params_billion,\n        \"n_questions\": len(QA_PAIRS),\n        \"duration_sec\": round(elapsed, 2),\n        \"energy_kwh\": tracker_energy_kwh(tracker),\n        \"emissions_kg_co2eq\": emissions_kg,\n        \"accuracy\": accuracy,\n    },\n    {\n        \"config\": \"closed-book-fp16\",\n        \"model\": MODEL_NAME,\n        \"params_billion\": n_params_billion,\n        \"n_questions\": len(QA_PAIRS),\n        \"duration_sec\": round(elapsed_closed, 2),\n        \"energy_kwh\": tracker_energy_kwh(tracker_closed),\n        \"emissions_kg_co2eq\": emissions_closed_kg,\n        \"accuracy\": accuracy_closed,\n    },\n])\nleaderboard.to_csv(\"leaderboard.csv\", index=False)\nleaderboard",
}
_fill_in = set(_solutions) if fill_in_solutions else set()


class ExerciseGaps(Exception):
    """A cell still has gaps marked ___."""


def _has_gap(lines):
    """True if `___` appears in the cell as code (not in a string or a comment)."""
    code = "".join(l for l in lines if not l.lstrip().startswith(("%", "!")))
    try:
        return any(t.type == tokenize.NAME and t.string == "___"
                   for t in tokenize.generate_tokens(io.StringIO(code).readline))
    except (tokenize.TokenError, SyntaxError):
        return False


def _fill_in_transformer(lines):
    """An exercise cell starts with `# Exercise N`; with fill-in on for N, it becomes the solution.
    A cell that still has gaps ___ stops with a message. (In IPython, ___ is a built-in name, the
    third-to-last output, so without this check the cell would run on with a wrong value.)"""
    m = re.match(r"# Exercise (\d+)\s*$", lines[0]) if lines else None
    if m and int(m.group(1)) in _fill_in:
        n = int(m.group(1))
        return [f"get_ipython().set_next_input(_solutions[{n}], replace=True)\n"] + (_solutions[n] + "\n").splitlines(True)
    if _has_gap(lines):
        return ['raise ExerciseGaps("this cell still has gaps marked ___: fill them in and run it again, '
                'or tick fill_in_solutions in the first code cell and run that cell")\n']
    return lines


_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup
                                     if getattr(t, "__name__", "") != "_fill_in_transformer"]
_ip.input_transformers_cleanup.append(_fill_in_transformer)
if fill_in_solutions:
    print("✓ Every exercise cell you run from now on gets its solution.")

## Step 0 — Install and import

This installs the libraries we need: `transformers` (to load and run the model),
`accelerate` (to place the model on the GPU efficiently), and `codecarbon` (to measure
energy use and estimate CO2 emissions for the code that runs while it's active).

In [ ]:
!pip install -q transformers accelerate codecarbon pandas

In [ ]:
import time
import textwrap
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from codecarbon import EmissionsTracker

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — the notebook will still run, just slower. "
          "Double check Runtime > Change runtime type > T4 GPU.")

## Step 1 — The corpus

This is our little "knowledge base": eight short documents about Green NLP itself
(fittingly). Our pipeline's job is to answer questions using *only* this text — no
outside knowledge allowed, even if the model happens to already know the answer.

Each document below is original text written for this course, summarizing findings from
real papers, cited at the end of each document.

In [ ]:
CORPUS = {
    "training_vs_inference": """
Training a large language model is a one-time, very expensive event: the organization
that builds the model spends a large amount of compute once. Inference — actually using
the trained model to answer a query — is comparatively cheap per request, but it happens
every single time anyone uses the model. Once a model is deployed and used by many people,
the accumulated cost of inference tends to dominate the model's total lifetime footprint,
even though any single inference looks negligible next to the training run.
Source: Patterson et al., "Carbon Emissions and Large Neural Network Training," 2021
(arXiv:2104.10350). Training GPT-3 was estimated to emit 552 metric tons of CO2-equivalent,
using about 1,287 megawatt-hours of electricity.
""",
    "data_centers": """
Data centers are the physical facilities that house the servers running most of the
world's AI models. According to the International Energy Agency, global data center
electricity consumption was about 415 terawatt-hours in 2024, roughly 1.5 percent of all
electricity consumed worldwide that year. That demand is growing fast: electricity use by
data centers rose by about 17 percent in 2025 alone, driven mostly by AI workloads, and is
projected to approach 950 terawatt-hours by 2030, or about 3 percent of global electricity
demand.
Source: International Energy Agency, "Energy and AI" and related data centre electricity
reports, 2024-2025.
""",
    "quantization_basics": """
Neural network weights are normally stored as 32-bit floating point numbers (FP32).
Quantization reduces the number of bits used to represent each weight — for example to
16-bit (FP16/BF16), 8-bit integers (INT8), or even lower. Fewer bits per number means a
smaller model file and less memory traffic when running it, at the cost of some numerical
precision. A useful analogy: quantizing a model is like reducing the number of colors used
to represent an image — the picture is still recognizable, but some fine detail is lost.
Whether this actually saves energy in practice depends on the hardware: it helps most when
the bottleneck is moving data in and out of memory, not raw computation.
Source: Grootendorst, "A Visual Guide to Quantization," 2024.
""",
    "measuring_carbon": """
Estimating the carbon footprint of running a model requires several assumptions: how much
electricity the hardware draws, the Power Usage Effectiveness (PUE) of the data center
(extra power spent on cooling and overhead), and the carbon intensity of the local
electricity grid — how much CO2 is emitted per unit of electricity, which varies a lot by
region and by time of day. Because of all these assumptions, published carbon estimates can
vary widely, and some influential early estimates have later been revised — sometimes by an
order of magnitude — once other researchers re-examined the methodology. This is one reason
this course emphasizes measuring your own numbers rather than only citing others' headline
figures.
""",
    "inference_cost_task_type": """
Not all models cost the same to run for the same task. A 2024 study compared task-specific
models (built and trained for one narrow job) against multi-purpose generative models (like
general-purpose chatbots) performing the exact same tasks. For extractive question
answering, the most efficient task-specific models emitted around 0.3 grams of
CO2-equivalent per 1,000 inferences, while general-purpose generative models emitted around
10 grams of CO2-equivalent per 1,000 inferences for the same task — roughly 33 times more.
The gap was consistent across several other tasks too, even when controlling for the number
of parameters.
Source: Luccioni, Jernite & Strubell, "Power Hungry Processing: Watts Driving the Cost of AI
Deployment?", FAccT 2024 (arXiv:2311.16863).
""",
    "rag_concept": """
Retrieval-Augmented Generation, or RAG, is a technique where a model is given relevant
external text — retrieved from a document collection — as part of its input at query time,
instead of relying only on what it "learned" during training. This lets a smaller, cheaper
model answer questions about information it was never trained on, without the cost of
retraining or fine-tuning a much larger model on that information. The retrieval step
usually uses a separate, lightweight embedding model to find which documents are relevant
before the language model ever sees them.
""",
    "small_models_trend": """
Between 2024 and 2026, there has been a strong trend toward small, efficient open-weight
language models (often called SLMs, small language models) in the 1-4 billion parameter
range, designed to run on a single consumer GPU or even a laptop CPU after quantization.
These models are trained to be very capable for their size, and combined with quantization
and retrieval, they let many real applications avoid depending on a large, general-purpose
model at all — often at a fraction of the inference cost, if the task doesn't need the
breadth of a much bigger model.
""",
    "transformer_basics": """
Modern language models are built from transformer layers. Inside each layer,
"self-attention" lets every position in the input sequence gather information from every
other position, which is what allows the model to relate words like "it" back to the noun
it refers to several words earlier. This mechanism is powerful but has a computational cost
that grows faster than the length of the input (roughly quadratically with sequence
length), and more parameters mean more numbers to multiply and move through memory for
every single token generated. This is the underlying reason why both model size and input
length are direct levers on energy cost.
""",
}

for name, text in CORPUS.items():
    print(f"--- {name} ({len(text.split())} words) ---")
print(f"\nTotal documents: {len(CORPUS)}")

## Step 2 — The quiz

These are the eight questions your pipeline needs to answer correctly, each with the short
reference answers we'll use to check quality automatically. This is a simple keyword
check, on whole words — good enough to compare configurations against each other in
this course.

In [ ]:
QA_PAIRS = [
    {
        "question": "According to Patterson et al. (2021), how many metric tons of CO2-equivalent did training GPT-3 emit?",
        "reference": ["552"],
    },
    {
        "question": "Per the IEA, roughly what percentage of global electricity consumption did data centers represent in 2024?",
        "reference": ["1.5"],
    },
    {
        "question": "What does the acronym PUE stand for?",
        "reference": ["power usage effectiveness"],
    },
    {
        "question": "What does carbon footprint estimation depend on besides hardware power draw and grid carbon intensity?",
        "reference": ["pue", "power usage effectiveness"],
    },
    {
        "question": "In the 2024 study on inference costs, about how many grams of CO2-equivalent do the most efficient task-specific models emit per 1,000 extractive question-answering inferences?",
        "reference": ["0.3"],
    },
    {
        "question": "What is the name of the technique where a model is given retrieved external text at query time instead of relying only on what it learned during training?",
        "reference": ["retrieval-augmented generation", "rag"],
    },
    {
        "question": "What size range (in billions of parameters) do small language models (SLMs) typically fall into?",
        "reference": ["1-4", "1 to 4", "1 and 4"],
    },
    {
        "question": "What mechanism inside a transformer layer lets every position gather information from every other position?",
        "reference": ["self-attention", "attention mechanism"],
    },
]

print(f"Number of questions: {len(QA_PAIRS)}")

## Step 3 — Load the baseline model

We'll use **Qwen2.5-1.5B-Instruct** (Apache 2.0 license, no gated access needed). It's small
enough to load quickly and run comfortably on a free-tier Colab GPU.

The first time you run this cell, it downloads about 3GB of model weights, which can take a
few minutes depending on Colab's connection that day — this is normal, just let it finish.

*Optional stretch goal:* once everything works, you can change `MODEL_NAME`
below to a bigger model (e.g. `Qwen/Qwen2.5-3B-Instruct`) and re-run, to see how the
baseline numbers change with size alone.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
print(f"Loaded {MODEL_NAME}")
print(f"Parameter count: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## Step 4 — The pipeline: stuff everything into context

This pipeline is deliberately the *naive* baseline: for every question, we hand the
model **all eight documents** plus the question, and let it find the answer itself. No
retrieval, no filtering — simple, and exactly the kind of "just throw the whole context at
a big model" approach that targeted retrieval is built to improve on.

### ✏️ Exercise 1 — Build the prompt and keep only the answer

**Goal:** see exactly what the model reads (all eight documents and the question), and how its output is cut from what it returns.

Two gaps:

1. In `build_prompt`, join all documents into one `context` string: each document as
   `[name]`, a newline, and its text, with a blank line between documents.
2. In `answer_question`, `model.generate` returns the prompt tokens **followed by** the new
   tokens. Keep only the new ones.

<details><summary><b>Hints</b> (click to open)</summary>

- `text.strip()` removes the blank lines around each document.
- `inputs["input_ids"].shape[1]` is the number of tokens in the prompt.

</details>

In [ ]:
# Exercise 1
def build_prompt(question, corpus):
    context = ___  # TODO: every document as "[name]\ntext", separated by blank lines
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise assistant. Answer the question using ONLY the "
                "documents provided below. Answer in one short sentence."
            ),
        },
        {
            "role": "user",
            "content": f"Documents:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def answer_question(question, corpus, max_new_tokens=60):
    prompt = build_prompt(question, corpus)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = ___  # TODO: only the newly generated token ids
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


sample_answer = answer_question(QA_PAIRS[0]["question"], CORPUS)
print("Q:", QA_PAIRS[0]["question"])
print("A:", sample_answer)

In [ ]:
# Check for Exercise 1. Run this cell as is.
_p = build_prompt("TEST QUESTION", CORPUS)
assert all(f"[{name}]" in _p for name in CORPUS), "every document name should appear in the prompt"
assert CORPUS["rag_concept"].strip() in _p, "document text should appear in the prompt"
assert "TEST QUESTION" in _p
assert isinstance(sample_answer, str) and sample_answer, "empty answer"
assert "Documents:" not in sample_answer, "the answer still contains the prompt: slice off the prompt tokens"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
def build_prompt(question, corpus):
    context = "\n\n".join(f"[{name}]\n{text.strip()}" for name, text in corpus.items())
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise assistant. Answer the question using ONLY the "
                "documents provided below. Answer in one short sentence."
            ),
        },
        {
            "role": "user",
            "content": f"Documents:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def answer_question(question, corpus, max_new_tokens=60):
    prompt = build_prompt(question, corpus)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


sample_answer = answer_question(QA_PAIRS[0]["question"], CORPUS)
print("Q:", QA_PAIRS[0]["question"])
print("A:", sample_answer)
```

</details>

In [ ]:
# Stuck on Exercise 1? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 1" above; your own cell stays as it is.)
get_ipython().set_next_input("def build_prompt(question, corpus):\n    context = \"\\n\\n\".join(f\"[{name}]\\n{text.strip()}\" for name, text in corpus.items())\n    messages = [\n        {\n            \"role\": \"system\",\n            \"content\": (\n                \"You are a concise assistant. Answer the question using ONLY the \"\n                \"documents provided below. Answer in one short sentence.\"\n            ),\n        },\n        {\n            \"role\": \"user\",\n            \"content\": f\"Documents:\\n{context}\\n\\nQuestion: {question}\",\n        },\n    ]\n    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n\ndef answer_question(question, corpus, max_new_tokens=60):\n    prompt = build_prompt(question, corpus)\n    inputs = tokenizer(prompt, return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        output = model.generate(\n            **inputs,\n            max_new_tokens=max_new_tokens,\n            do_sample=False,\n        )\n    generated = output[0][inputs[\"input_ids\"].shape[1]:]\n    return tokenizer.decode(generated, skip_special_tokens=True).strip()\n\n\nsample_answer = answer_question(QA_PAIRS[0][\"question\"], CORPUS)\nprint(\"Q:\", QA_PAIRS[0][\"question\"])\nprint(\"A:\", sample_answer)")


## Step 5 — Measure: run the full quiz with CodeCarbon

Now we run all eight questions through the pipeline, with `EmissionsTracker` running the
whole time. It estimates energy use (kWh) and CO2-equivalent (kg) for the code that runs
inside the `tracker.start()` / `tracker.stop()` block.

**Known limitation, worth noticing:** CodeCarbon *estimates* emissions — it doesn't have a
wattmeter plugged into the wall. On some GPUs (including some Colab allocations) it can't
read power draw directly and falls back to a rougher estimate. If you see a warning about
an unrecognized GPU, that's this limitation in action, a live example of
the uncertainty described in the `measuring_carbon` document above.

You'll likely see two more CodeCarbon warnings later in this notebook, for the same
reason — both harmless:

- `Multiple instances of codecarbon are allowed to run at the same time` — this notebook
  creates a fresh tracker for almost every measurement; CodeCarbon is just noting that.
- `Unable to access geographical location ... falls back to using the backup API` —
  CodeCarbon tries to look up your local electricity grid's carbon intensity from your IP;
  if that lookup fails (campus network, VPN, the geolocation service being down) it falls
  back to a default intensity value instead. The *relative* comparisons this notebook
  makes (which approach is cheaper than which) stay valid either way, since the same
  fallback applies consistently to every measurement — but the *absolute* g CO2eq numbers
  are then less geographically precise — another live example of measurement uncertainty.

### ✏️ Exercise 2 — Measure the run

**Goal:** wrap a piece of work in a CodeCarbon tracker and a clock, the pattern every measurement in this course uses.

Start the CodeCarbon tracker and a clock before the loop. After the loop, store the
elapsed seconds in `elapsed` and the tracker's result in `emissions_kg`.

<details><summary><b>Hints</b> (click to open)</summary>

- `tracker.start()` starts measuring; `tracker.stop()` stops and returns kg CO2eq.
- `time.time()` is the current time in seconds.

</details>

In [ ]:
# Exercise 2
tracker = EmissionsTracker(
    project_name="green-nlp-baseline",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results = []
# TODO: start the tracker and the clock
___

for pair in QA_PAIRS:
    answer = answer_question(pair["question"], CORPUS)
    results.append({"question": pair["question"], "reference": pair["reference"], "answer": answer})

# TODO: stop measuring; store elapsed (seconds) and emissions_kg
___

print(f"Ran {len(QA_PAIRS)} questions in {elapsed:.1f} seconds")
print(f"Estimated emissions: {emissions_kg * 1000:.4f} g CO2eq")

In [ ]:
# Check for Exercise 2. Run this cell as is.
assert len(results) == len(QA_PAIRS)
assert 0 < elapsed < 3600, elapsed
assert emissions_kg is not None and emissions_kg > 0, emissions_kg
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
tracker = EmissionsTracker(
    project_name="green-nlp-baseline",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results = []
tracker.start()
start_time = time.time()

for pair in QA_PAIRS:
    answer = answer_question(pair["question"], CORPUS)
    results.append({"question": pair["question"], "reference": pair["reference"], "answer": answer})

elapsed = time.time() - start_time
emissions_kg = tracker.stop()

print(f"Ran {len(QA_PAIRS)} questions in {elapsed:.1f} seconds")
print(f"Estimated emissions: {emissions_kg * 1000:.4f} g CO2eq")
```

</details>

In [ ]:
# Stuck on Exercise 2? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 2" above; your own cell stays as it is.)
get_ipython().set_next_input("tracker = EmissionsTracker(\n    project_name=\"green-nlp-baseline\",\n    measure_power_secs=1,\n    save_to_file=False,\n    log_level=\"warning\",\n)\n\nresults = []\ntracker.start()\nstart_time = time.time()\n\nfor pair in QA_PAIRS:\n    answer = answer_question(pair[\"question\"], CORPUS)\n    results.append({\"question\": pair[\"question\"], \"reference\": pair[\"reference\"], \"answer\": answer})\n\nelapsed = time.time() - start_time\nemissions_kg = tracker.stop()\n\nprint(f\"Ran {len(QA_PAIRS)} questions in {elapsed:.1f} seconds\")\nprint(f\"Estimated emissions: {emissions_kg * 1000:.4f} g CO2eq\")")


## Step 6 — Score quality

A simple, transparent scoring rule. An answer counts as right when it contains one of the question's references as whole words. The answer and the reference are read the same way first: lowercased and cut into words and numbers. So `rag` counts in “(RAG)” but not inside “average” or “storage”, and `0.3` does not count inside “10.3”. Between letters, spaces, hyphens and dots do not count: “Self-Attention”, “self attention” and “self-attention” all match `self-attention`. Numbers keep their decimal point, and a thousands comma does not count: “1,287” matches `1287`, and “1.5%” contains `1.5`. A quick proxy, good enough to
compare configurations against each other consistently. It misses a correct output
worded unlike every reference.

### ✏️ Exercise 3 — Score the answers

**Goal:** turn eight outputs into one quality number that later runs can be compared with, and see what a keyword check can and cannot tell.

Write `score_answer`: an answer is correct if it contains at least one of the
references as whole words, ignoring upper and lower case. The given helper
`contains_reference` does the whole-word test for one reference (Step 6 above says how it reads words). Then compute `accuracy`, the fraction
of correct answers.

<details><summary><b>Hints</b> (click to open)</summary>

- `contains_reference` lowercases both sides itself.
- `contains_reference(answer, ref)` finds whole words only: `"rag"` in `"(RAG)"`, not in `"storage"`.
- In Python, `True` counts as 1 in `sum`.

</details>

In [ ]:
# Exercise 3
import re


def words(text):
    """The words and numbers of a text, lowercased: "1,287" becomes 1287, "1.5%" becomes 1.5 and %."""
    text = re.sub(r"(?<=\d),(?=\d{3}(?!\d))", "", text.lower())
    return re.findall(r"\d+(?:\.\d+)?|%|[^\W\d_]+", text)


def joined(ws):
    """The words as one string: letters run together, a number keeps a space on each side."""
    out = ws[0] if ws else ""
    for prev, w in zip(ws, ws[1:]):
        out += w if prev.isalpha() and w.isalpha() else " " + w
    return out


def contains_reference(answer, reference):
    """True if some run of whole words in the answer reads the same as the reference."""
    a, target = words(answer), joined(words(reference))
    return any(joined(a[i:j]) == target for i in range(len(a)) for j in range(i + 1, len(a) + 1))


def score_answer(answer, reference_list):
    # TODO: True if the answer contains at least one reference as whole words
    # (use contains_reference)
    ___


for r in results:
    r["correct"] = score_answer(r["answer"], r["reference"])

accuracy = ___  # TODO: fraction of results that are correct

for r in results:
    mark = "✓" if r["correct"] else "✗"
    print(f"{mark} Q: {r['question']}")
    print(f"   A: {r['answer']}\n")

print(f"Accuracy: {accuracy:.0%} ({sum(r['correct'] for r in results)}/{len(results)})")

In [ ]:
# Check for Exercise 3. Run this cell as is.
assert score_answer("It emitted 552 tonnes.", ["552"]) is True
assert score_answer("Power Usage Effectiveness", ["power usage effectiveness"]) is True
assert score_answer("I do not know.", ["552"]) is False
assert score_answer("It is kept in storage.", ["rag"]) is False, "match whole words only: 'rag' is not in 'storage'"
assert score_answer("About 10.3 grams.", ["0.3"]) is False, "a number counts whole: 0.3 is not in 10.3"
assert score_answer("Self attention.", ["self-attention"]) is True, "a space or hyphen between letters does not count"
assert abs(accuracy - sum(score_answer(r["answer"], r["reference"]) for r in results) / len(results)) < 1e-9
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
import re


def words(text):
    """The words and numbers of a text, lowercased: "1,287" becomes 1287, "1.5%" becomes 1.5 and %."""
    text = re.sub(r"(?<=\d),(?=\d{3}(?!\d))", "", text.lower())
    return re.findall(r"\d+(?:\.\d+)?|%|[^\W\d_]+", text)


def joined(ws):
    """The words as one string: letters run together, a number keeps a space on each side."""
    out = ws[0] if ws else ""
    for prev, w in zip(ws, ws[1:]):
        out += w if prev.isalpha() and w.isalpha() else " " + w
    return out


def contains_reference(answer, reference):
    """True if some run of whole words in the answer reads the same as the reference."""
    a, target = words(answer), joined(words(reference))
    return any(joined(a[i:j]) == target for i in range(len(a)) for j in range(i + 1, len(a) + 1))


def score_answer(answer, reference_list):
    return any(contains_reference(answer, ref) for ref in reference_list)


for r in results:
    r["correct"] = score_answer(r["answer"], r["reference"])

accuracy = sum(r["correct"] for r in results) / len(results)

for r in results:
    mark = "✓" if r["correct"] else "✗"
    print(f"{mark} Q: {r['question']}")
    print(f"   A: {r['answer']}\n")

print(f"Accuracy: {accuracy:.0%} ({sum(r['correct'] for r in results)}/{len(results)})")
```

</details>

In [ ]:
# Stuck on Exercise 3? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 3" above; your own cell stays as it is.)
get_ipython().set_next_input("import re\n\n\ndef words(text):\n    \"\"\"The words and numbers of a text, lowercased: \"1,287\" becomes 1287, \"1.5%\" becomes 1.5 and %.\"\"\"\n    text = re.sub(r\"(?<=\\d),(?=\\d{3}(?!\\d))\", \"\", text.lower())\n    return re.findall(r\"\\d+(?:\\.\\d+)?|%|[^\\W\\d_]+\", text)\n\n\ndef joined(ws):\n    \"\"\"The words as one string: letters run together, a number keeps a space on each side.\"\"\"\n    out = ws[0] if ws else \"\"\n    for prev, w in zip(ws, ws[1:]):\n        out += w if prev.isalpha() and w.isalpha() else \" \" + w\n    return out\n\n\ndef contains_reference(answer, reference):\n    \"\"\"True if some run of whole words in the answer reads the same as the reference.\"\"\"\n    a, target = words(answer), joined(words(reference))\n    return any(joined(a[i:j]) == target for i in range(len(a)) for j in range(i + 1, len(a) + 1))\n\n\ndef score_answer(answer, reference_list):\n    return any(contains_reference(answer, ref) for ref in reference_list)\n\n\nfor r in results:\n    r[\"correct\"] = score_answer(r[\"answer\"], r[\"reference\"])\n\naccuracy = sum(r[\"correct\"] for r in results) / len(results)\n\nfor r in results:\n    mark = \"✓\" if r[\"correct\"] else \"✗\"\n    print(f\"{mark} Q: {r['question']}\")\n    print(f\"   A: {r['answer']}\\n\")\n\nprint(f\"Accuracy: {accuracy:.0%} ({sum(r['correct'] for r in results)}/{len(results)})\")")


### Step 6, continued — the same quiz with no documents

Every prompt above contains all eight documents, and the documents contain the answers. So
the quiz measures whether the model can *find* a fact it has been handed. How much does the
model know without them? Here is the same quiz, same model and same one-sentence
instruction, with the question alone. This is called **closed-book** question answering:
the model can only draw on what it learned in training.

### ✏️ Exercise 4 — The closed-book prompt

**Goal:** take the documents out of the input, to measure what they contribute to accuracy and to cost.

Complete the user message for `answer_closed_book`: the question alone, with no
documents.

<details><summary><b>Hints</b> (click to open)</summary>

- Compare with the user message in `build_prompt` (Exercise 1).
- An f-string: `f"Question: {question}"`.

</details>

In [ ]:
# Exercise 4
def answer_closed_book(question, max_new_tokens=60):
    messages = [
        {"role": "system",
         "content": "You are a concise assistant. Answer the question in one short sentence."},
        ___,  # TODO: the user message: "Question: " followed by the question
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_closed = EmissionsTracker(
    project_name="green-nlp-closed-book",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results_closed = []
tracker_closed.start()
start_time = time.time()

for pair in QA_PAIRS:
    answer = answer_closed_book(pair["question"])
    results_closed.append({"question": pair["question"], "answer": answer,
                           "correct": score_answer(answer, pair["reference"])})

elapsed_closed = time.time() - start_time
emissions_closed_kg = tracker_closed.stop()
accuracy_closed = sum(r["correct"] for r in results_closed) / len(results_closed)

for r_docs, r_closed in zip(results, results_closed):
    print(f"Q: {r_docs['question']}")
    print(f"   with documents {'✓' if r_docs['correct'] else '✗'}  {r_docs['answer']}")
    print(f"   no documents   {'✓' if r_closed['correct'] else '✗'}  {r_closed['answer']}\n")

tokens_with_docs = len(tokenizer(build_prompt(QA_PAIRS[0]["question"], CORPUS))["input_ids"])
print(f"Accuracy with documents: {accuracy:.0%}   no documents: {accuracy_closed:.0%}")
print(f"Emissions with documents: {emissions_kg * 1000:.4f} g CO2eq   "
      f"no documents: {emissions_closed_kg * 1000:.4f} g CO2eq")
print(f"Prompt length for the first question: {tokens_with_docs} tokens with documents")

In [ ]:
# Check for Exercise 4. Run this cell as is.
assert len(results_closed) == len(QA_PAIRS)
assert all(isinstance(r["answer"], str) and r["answer"] for r in results_closed)
assert "Documents:" not in results_closed[0]["answer"]
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
def answer_closed_book(question, max_new_tokens=60):
    messages = [
        {"role": "system",
         "content": "You are a concise assistant. Answer the question in one short sentence."},
        {"role": "user", "content": f"Question: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_closed = EmissionsTracker(
    project_name="green-nlp-closed-book",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results_closed = []
tracker_closed.start()
start_time = time.time()

for pair in QA_PAIRS:
    answer = answer_closed_book(pair["question"])
    results_closed.append({"question": pair["question"], "answer": answer,
                           "correct": score_answer(answer, pair["reference"])})

elapsed_closed = time.time() - start_time
emissions_closed_kg = tracker_closed.stop()
accuracy_closed = sum(r["correct"] for r in results_closed) / len(results_closed)

for r_docs, r_closed in zip(results, results_closed):
    print(f"Q: {r_docs['question']}")
    print(f"   with documents {'✓' if r_docs['correct'] else '✗'}  {r_docs['answer']}")
    print(f"   no documents   {'✓' if r_closed['correct'] else '✗'}  {r_closed['answer']}\n")

tokens_with_docs = len(tokenizer(build_prompt(QA_PAIRS[0]["question"], CORPUS))["input_ids"])
print(f"Accuracy with documents: {accuracy:.0%}   no documents: {accuracy_closed:.0%}")
print(f"Emissions with documents: {emissions_kg * 1000:.4f} g CO2eq   "
      f"no documents: {emissions_closed_kg * 1000:.4f} g CO2eq")
print(f"Prompt length for the first question: {tokens_with_docs} tokens with documents")
```

</details>

In [ ]:
# Stuck on Exercise 4? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 4" above; your own cell stays as it is.)
get_ipython().set_next_input("def answer_closed_book(question, max_new_tokens=60):\n    messages = [\n        {\"role\": \"system\",\n         \"content\": \"You are a concise assistant. Answer the question in one short sentence.\"},\n        {\"role\": \"user\", \"content\": f\"Question: {question}\"},\n    ]\n    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs = tokenizer(prompt, return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)\n    generated = output[0][inputs[\"input_ids\"].shape[1]:]\n    return tokenizer.decode(generated, skip_special_tokens=True).strip()\n\n\ntracker_closed = EmissionsTracker(\n    project_name=\"green-nlp-closed-book\",\n    measure_power_secs=1,\n    save_to_file=False,\n    log_level=\"warning\",\n)\n\nresults_closed = []\ntracker_closed.start()\nstart_time = time.time()\n\nfor pair in QA_PAIRS:\n    answer = answer_closed_book(pair[\"question\"])\n    results_closed.append({\"question\": pair[\"question\"], \"answer\": answer,\n                           \"correct\": score_answer(answer, pair[\"reference\"])})\n\nelapsed_closed = time.time() - start_time\nemissions_closed_kg = tracker_closed.stop()\naccuracy_closed = sum(r[\"correct\"] for r in results_closed) / len(results_closed)\n\nfor r_docs, r_closed in zip(results, results_closed):\n    print(f\"Q: {r_docs['question']}\")\n    print(f\"   with documents {'✓' if r_docs['correct'] else '✗'}  {r_docs['answer']}\")\n    print(f\"   no documents   {'✓' if r_closed['correct'] else '✗'}  {r_closed['answer']}\\n\")\n\ntokens_with_docs = len(tokenizer(build_prompt(QA_PAIRS[0][\"question\"], CORPUS))[\"input_ids\"])\nprint(f\"Accuracy with documents: {accuracy:.0%}   no documents: {accuracy_closed:.0%}\")\nprint(f\"Emissions with documents: {emissions_kg * 1000:.4f} g CO2eq   \"\n      f\"no documents: {emissions_closed_kg * 1000:.4f} g CO2eq\")\nprint(f\"Prompt length for the first question: {tokens_with_docs} tokens with documents\")")


### ✏️ Exercise 5 — Estimate the compute from an equation (advanced)

**Goal:** predict the compute of a run from the number of parameters and tokens, and compare the prediction with what you measured.

A forward pass through a transformer costs about **2N floating-point operations
(FLOPs) per token**, where N is the number of parameters: each weight is used in one
multiplication and one addition (Kaplan et al. 2020, *Scaling Laws for Neural Language
Models*, arXiv:2001.08361). The model processes every prompt token once, and then every
generated token once, so for one question:

FLOPs ≈ 2 · N · (prompt tokens + generated tokens)

Complete the three functions, then run the cell. It estimates the compute of the run with
documents (`results`) and of the closed-book run (`results_closed`), and prints the ratio of
the two next to the ratios you measured. Count the generated tokens by tokenizing the
stored answer; this is close to, but not always exactly, what the model generated.

**Reflection:** do the estimated and the measured ratios agree? The equation counts
arithmetic only. What else takes time when a model generates text one token at a time?

<details><summary><b>Hints</b> (click to open)</summary>

- `tokenizer(text)["input_ids"]` is the list of token ids for `text`.
- For each result: `count_tokens(make_prompt(r["question"])) + count_tokens(r["answer"])`.
- `total += forward_flops(n_tokens)`.

</details>

In [ ]:
# Exercise 5
N_PARAMS = sum(p.numel() for p in model.parameters())


def closed_book_prompt(question):
    messages = [
        {"role": "system",
         "content": "You are a concise assistant. Answer the question in one short sentence."},
        {"role": "user", "content": f"Question: {question}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def count_tokens(text):
    # TODO: the number of tokens in text, using the model's tokenizer
    ___


def forward_flops(n_tokens, n_params=N_PARAMS):
    # TODO: the 2N-per-token estimate for n_tokens tokens
    ___


def run_flops(results_list, make_prompt):
    """Estimated FLOPs of a whole run. make_prompt(question) returns the prompt text."""
    total = 0
    for r in results_list:
        # TODO: tokens processed for this question (prompt + answer); add their FLOPs to total
        ___
    return total

flops_docs = run_flops(results, lambda q: build_prompt(q, CORPUS))
flops_closed = run_flops(results_closed, closed_book_prompt)

print(f"With documents: {flops_docs:.2e} FLOPs")
print(f"Closed-book:    {flops_closed:.2e} FLOPs")
print(f"Ratio, estimated FLOPs:  {flops_docs / flops_closed:5.1f}x")
print(f"Ratio, measured time:    {elapsed / elapsed_closed:5.1f}x")
print(f"Ratio, measured CO2eq:   {emissions_kg / emissions_closed_kg:5.1f}x")

In [ ]:
# Check for Exercise 5. Run this cell as is.
_q = QA_PAIRS[0]["question"]
assert count_tokens(build_prompt(_q, CORPUS)) == len(tokenizer(build_prompt(_q, CORPUS))["input_ids"])
assert forward_flops(1000, n_params=1e9) == 2e12, "1000 tokens through 1B parameters: 2e12 FLOPs"
_expected = sum(2 * N_PARAMS * (count_tokens(build_prompt(r["question"], CORPUS)) + count_tokens(r["answer"]))
                for r in results)
assert abs(flops_docs - _expected) / _expected < 1e-9, "count the prompt tokens and the answer tokens"
assert flops_docs > flops_closed, "the prompts with documents are much longer"
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
N_PARAMS = sum(p.numel() for p in model.parameters())


def closed_book_prompt(question):
    messages = [
        {"role": "system",
         "content": "You are a concise assistant. Answer the question in one short sentence."},
        {"role": "user", "content": f"Question: {question}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def count_tokens(text):
    return len(tokenizer(text)["input_ids"])


def forward_flops(n_tokens, n_params=N_PARAMS):
    return 2 * n_params * n_tokens


def run_flops(results_list, make_prompt):
    """Estimated FLOPs of a whole run. make_prompt(question) returns the prompt text."""
    total = 0
    for r in results_list:
        n_tokens = count_tokens(make_prompt(r["question"])) + count_tokens(r["answer"])
        total += forward_flops(n_tokens)
    return total

flops_docs = run_flops(results, lambda q: build_prompt(q, CORPUS))
flops_closed = run_flops(results_closed, closed_book_prompt)

print(f"With documents: {flops_docs:.2e} FLOPs")
print(f"Closed-book:    {flops_closed:.2e} FLOPs")
print(f"Ratio, estimated FLOPs:  {flops_docs / flops_closed:5.1f}x")
print(f"Ratio, measured time:    {elapsed / elapsed_closed:5.1f}x")
print(f"Ratio, measured CO2eq:   {emissions_kg / emissions_closed_kg:5.1f}x")
```

</details>

In [ ]:
# Stuck on Exercise 5? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 5" above; your own cell stays as it is.)
get_ipython().set_next_input("N_PARAMS = sum(p.numel() for p in model.parameters())\n\n\ndef closed_book_prompt(question):\n    messages = [\n        {\"role\": \"system\",\n         \"content\": \"You are a concise assistant. Answer the question in one short sentence.\"},\n        {\"role\": \"user\", \"content\": f\"Question: {question}\"},\n    ]\n    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n\ndef count_tokens(text):\n    return len(tokenizer(text)[\"input_ids\"])\n\n\ndef forward_flops(n_tokens, n_params=N_PARAMS):\n    return 2 * n_params * n_tokens\n\n\ndef run_flops(results_list, make_prompt):\n    \"\"\"Estimated FLOPs of a whole run. make_prompt(question) returns the prompt text.\"\"\"\n    total = 0\n    for r in results_list:\n        n_tokens = count_tokens(make_prompt(r[\"question\"])) + count_tokens(r[\"answer\"])\n        total += forward_flops(n_tokens)\n    return total\n\nflops_docs = run_flops(results, lambda q: build_prompt(q, CORPUS))\nflops_closed = run_flops(results_closed, closed_book_prompt)\n\nprint(f\"With documents: {flops_docs:.2e} FLOPs\")\nprint(f\"Closed-book:    {flops_closed:.2e} FLOPs\")\nprint(f\"Ratio, estimated FLOPs:  {flops_docs / flops_closed:5.1f}x\")\nprint(f\"Ratio, measured time:    {elapsed / elapsed_closed:5.1f}x\")\nprint(f\"Ratio, measured CO2eq:   {emissions_kg / emissions_closed_kg:5.1f}x\")")


### ✏️ Exercise 6 — Read the documents once: prefix caching (advanced)

**Goal:** make the pipeline cheaper without changing its output, by reading the shared documents only once.

Every prompt in Step 5 starts with the same system message and the same eight
documents. Only the question at the end changes. Still, the model processes those shared
tokens again for each of the eight questions.

While a model reads a prompt, each attention layer computes a key vector and a value
vector for every token. They are kept in the **KV cache**, so that later tokens can attend
to them without computing them again. A cache built once for the shared start of the
prompt (the **prefix**) can be reused for every question. This is called **prefix
caching**.

1. `prefix_text` is the part of the prompt before `"Question: "`.
2. Inside the measured block, run the model once on the prefix and keep the cache.
3. In `answer_with_prefix_cache`, pass a **copy** of that cache to `model.generate`.
   `generate` then processes only the tokens that are not in the cache yet. It also adds
   the new tokens to the cache it is given, so each question needs its own copy.

The cell measures the new run like Step 5. The answers should match the baseline's. In
float16, the order of the arithmetic changes the last digits of some values, so an
answer or two may differ by a word.

**Reflection:** compare the saving in time with the share of FLOPs that the documents
account for in the previous exercise. Which part of answering a question does prefix
caching remove, and which part does it leave?

<details><summary><b>Hints</b> (click to open)</summary>

- `full_example.index("Question: ")` is the position where the question starts; slice up to it.
- `inputs["input_ids"][0, :prefix_ids.shape[1]]` are the prompt's first tokens; compare them with `torch.equal`.
- Filling the cache: `model(prefix_ids, past_key_values=prefix_cache, use_cache=True)`.
- `copy.deepcopy(prefix_cache)` makes a copy that `generate` can change.

</details>

In [ ]:
# Exercise 6
import copy
from transformers import DynamicCache

full_example = build_prompt("PLACEHOLDER", CORPUS)
prefix_text = ___  # TODO: the part of full_example before "Question: "
prefix_ids = tokenizer(prefix_text, return_tensors="pt")["input_ids"].to(model.device)


def answer_with_prefix_cache(question, max_new_tokens=60):
    inputs = tokenizer(build_prompt(question, CORPUS), return_tensors="pt").to(model.device)
    # TODO: assert that the prompt's first tokens are exactly prefix_ids
    ___
    with torch.no_grad():
        output = model.generate(
            **inputs,
            past_key_values=___,  # TODO: a copy of the prefix cache
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_cached = EmissionsTracker(
    project_name="green-nlp-prefix-cache",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results_cached = []
tracker_cached.start()
start_time = time.time()

prefix_cache = DynamicCache()
with torch.no_grad():
    # TODO: one forward pass over prefix_ids that fills prefix_cache
    # (pass past_key_values=prefix_cache and use_cache=True)
    ___

for pair in QA_PAIRS:
    results_cached.append({"question": pair["question"],
                           "answer": answer_with_prefix_cache(pair["question"])})

elapsed_cached = time.time() - start_time
emissions_cached_kg = tracker_cached.stop()

same = sum(r["answer"] == c["answer"] for r, c in zip(results, results_cached))
print(f"Prefix: {prefix_ids.shape[1]} tokens, computed once instead of {len(QA_PAIRS)} times")
print(f"Same answer as the baseline: {same}/{len(QA_PAIRS)}")
print(f"Baseline:       {elapsed:6.1f} s  {emissions_kg * 1000:.4f} g CO2eq")
print(f"Prefix caching: {elapsed_cached:6.1f} s  {emissions_cached_kg * 1000:.4f} g CO2eq")

In [ ]:
# Check for Exercise 6. Run this cell as is.
assert full_example.startswith(prefix_text) and full_example[len(prefix_text):].startswith("Question: "), \
    "prefix_text should be everything before \"Question: \""
assert prefix_cache.get_seq_length() == prefix_ids.shape[1], \
    "the prefix cache grew: pass a copy (copy.deepcopy) to generate, and fill it only once"
assert len(results_cached) == len(QA_PAIRS)
assert same >= len(QA_PAIRS) - 2, f"only {same} answers match the baseline: is the cache built from the right prefix?"
print("✓ Exercise 6 passed")

<details><summary><b>Solution to Exercise 6</b> (open it after you have tried)</summary>

```python
import copy
from transformers import DynamicCache

full_example = build_prompt("PLACEHOLDER", CORPUS)
prefix_text = full_example[:full_example.index("Question: ")]
prefix_ids = tokenizer(prefix_text, return_tensors="pt")["input_ids"].to(model.device)


def answer_with_prefix_cache(question, max_new_tokens=60):
    inputs = tokenizer(build_prompt(question, CORPUS), return_tensors="pt").to(model.device)
    assert torch.equal(inputs["input_ids"][0, :prefix_ids.shape[1]], prefix_ids[0])
    with torch.no_grad():
        output = model.generate(
            **inputs,
            past_key_values=copy.deepcopy(prefix_cache),
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_cached = EmissionsTracker(
    project_name="green-nlp-prefix-cache",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results_cached = []
tracker_cached.start()
start_time = time.time()

prefix_cache = DynamicCache()
with torch.no_grad():
    model(prefix_ids, past_key_values=prefix_cache, use_cache=True)

for pair in QA_PAIRS:
    results_cached.append({"question": pair["question"],
                           "answer": answer_with_prefix_cache(pair["question"])})

elapsed_cached = time.time() - start_time
emissions_cached_kg = tracker_cached.stop()

same = sum(r["answer"] == c["answer"] for r, c in zip(results, results_cached))
print(f"Prefix: {prefix_ids.shape[1]} tokens, computed once instead of {len(QA_PAIRS)} times")
print(f"Same answer as the baseline: {same}/{len(QA_PAIRS)}")
print(f"Baseline:       {elapsed:6.1f} s  {emissions_kg * 1000:.4f} g CO2eq")
print(f"Prefix caching: {elapsed_cached:6.1f} s  {emissions_cached_kg * 1000:.4f} g CO2eq")
```

</details>

In [ ]:
# Stuck on Exercise 6? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 6" above; your own cell stays as it is.)
get_ipython().set_next_input("import copy\nfrom transformers import DynamicCache\n\nfull_example = build_prompt(\"PLACEHOLDER\", CORPUS)\nprefix_text = full_example[:full_example.index(\"Question: \")]\nprefix_ids = tokenizer(prefix_text, return_tensors=\"pt\")[\"input_ids\"].to(model.device)\n\n\ndef answer_with_prefix_cache(question, max_new_tokens=60):\n    inputs = tokenizer(build_prompt(question, CORPUS), return_tensors=\"pt\").to(model.device)\n    assert torch.equal(inputs[\"input_ids\"][0, :prefix_ids.shape[1]], prefix_ids[0])\n    with torch.no_grad():\n        output = model.generate(\n            **inputs,\n            past_key_values=copy.deepcopy(prefix_cache),\n            max_new_tokens=max_new_tokens,\n            do_sample=False,\n        )\n    generated = output[0][inputs[\"input_ids\"].shape[1]:]\n    return tokenizer.decode(generated, skip_special_tokens=True).strip()\n\n\ntracker_cached = EmissionsTracker(\n    project_name=\"green-nlp-prefix-cache\",\n    measure_power_secs=1,\n    save_to_file=False,\n    log_level=\"warning\",\n)\n\nresults_cached = []\ntracker_cached.start()\nstart_time = time.time()\n\nprefix_cache = DynamicCache()\nwith torch.no_grad():\n    model(prefix_ids, past_key_values=prefix_cache, use_cache=True)\n\nfor pair in QA_PAIRS:\n    results_cached.append({\"question\": pair[\"question\"],\n                           \"answer\": answer_with_prefix_cache(pair[\"question\"])})\n\nelapsed_cached = time.time() - start_time\nemissions_cached_kg = tracker_cached.stop()\n\nsame = sum(r[\"answer\"] == c[\"answer\"] for r, c in zip(results, results_cached))\nprint(f\"Prefix: {prefix_ids.shape[1]} tokens, computed once instead of {len(QA_PAIRS)} times\")\nprint(f\"Same answer as the baseline: {same}/{len(QA_PAIRS)}\")\nprint(f\"Baseline:       {elapsed:6.1f} s  {emissions_kg * 1000:.4f} g CO2eq\")\nprint(f\"Prefix caching: {elapsed_cached:6.1f} s  {emissions_cached_kg * 1000:.4f} g CO2eq\")")


**Reflection:** Which questions did the model answer correctly with no documents, and
which did it miss? For the misses, did it say it didn't know, or did it give a specific
answer anyway? Compare the two accuracy numbers and the two emissions numbers: what do the
documents buy, and what do they cost?

<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

Without documents, the model can only use what it absorbed
in training. Widely known terms, such as what PUE stands for or what self-attention does,
are within reach. Specific figures from particular reports (552 tonnes, 1.5 percent, 0.3
grams) mostly are not, and a small model rarely answers "I don't know": it produces a
fluent, specific, wrong answer instead, such as a different tonnage for GPT-3 or a
plausible but wrong name for RAG. That is a hallucination, and it reads exactly like a
correct answer. The gap between the two accuracy numbers is what the documents contribute.
They cost something too: each prompt with documents is over a thousand tokens, against a
few dozen without. Emissions fall by less than that ratio, because writing the answer takes one run of the model per written token either way (only each run's attention to the prompt gets cheaper without documents), and closed-book answers tend to be longer.
Session 3 asks whether handing over only the relevant document keeps the accuracy while
cutting the prompt.

</details>

## Step 7 — Log these runs to your leaderboard

Two rows go into your leaderboard. `baseline-fp16` is the reference every later session
compares against: future rows use the same questions and scoring, changing one thing at a
time — a quantized version of this exact pipeline, then a retrieval-augmented one.
`closed-book-fp16` records the no-documents run, as a reference point for what the
documents contribute.

### ✏️ Exercise 7 — Log the closed-book row

**Goal:** record the closed-book run in the same format as the baseline, so later sessions can compare with it.

Fill in the measured values for the `closed-book-fp16` row.

<details><summary><b>Hints</b> (click to open)</summary>

- The closed-book run stored its results in variables ending in `_closed`.
- Look at the `baseline-fp16` row just above for the pattern.

</details>

In [ ]:
# Exercise 7
def tracker_energy_kwh(tracker_obj):
    try:
        return float(tracker_obj.final_emissions_data.energy_consumed)
    except (AttributeError, TypeError):
        return None


n_params_billion = round(sum(p.numel() for p in model.parameters()) / 1e9, 2)

leaderboard = pd.DataFrame([
    {
        "config": "baseline-fp16",
        "model": MODEL_NAME,
        "params_billion": n_params_billion,
        "n_questions": len(QA_PAIRS),
        "duration_sec": round(elapsed, 2),
        "energy_kwh": tracker_energy_kwh(tracker),
        "emissions_kg_co2eq": emissions_kg,
        "accuracy": accuracy,
    },
    {
        "config": "closed-book-fp16",
        "model": MODEL_NAME,
        "params_billion": n_params_billion,
        "n_questions": len(QA_PAIRS),
        "duration_sec": ___,  # TODO: closed-book duration, rounded to 2 decimals
        "energy_kwh": ___,  # TODO: closed-book energy, from its tracker
        "emissions_kg_co2eq": ___,
        "accuracy": ___,
    },
])
leaderboard.to_csv("leaderboard.csv", index=False)
leaderboard

In [ ]:
# Check for Exercise 7. Run this cell as is.
_row = leaderboard[leaderboard["config"] == "closed-book-fp16"].iloc[0]
assert _row["emissions_kg_co2eq"] == emissions_closed_kg
assert _row["accuracy"] == accuracy_closed
assert _row["duration_sec"] == round(elapsed_closed, 2)
print("✓ Exercise 7 passed")

<details><summary><b>Solution to Exercise 7</b> (open it after you have tried)</summary>

```python
def tracker_energy_kwh(tracker_obj):
    try:
        return float(tracker_obj.final_emissions_data.energy_consumed)
    except (AttributeError, TypeError):
        return None


n_params_billion = round(sum(p.numel() for p in model.parameters()) / 1e9, 2)

leaderboard = pd.DataFrame([
    {
        "config": "baseline-fp16",
        "model": MODEL_NAME,
        "params_billion": n_params_billion,
        "n_questions": len(QA_PAIRS),
        "duration_sec": round(elapsed, 2),
        "energy_kwh": tracker_energy_kwh(tracker),
        "emissions_kg_co2eq": emissions_kg,
        "accuracy": accuracy,
    },
    {
        "config": "closed-book-fp16",
        "model": MODEL_NAME,
        "params_billion": n_params_billion,
        "n_questions": len(QA_PAIRS),
        "duration_sec": round(elapsed_closed, 2),
        "energy_kwh": tracker_energy_kwh(tracker_closed),
        "emissions_kg_co2eq": emissions_closed_kg,
        "accuracy": accuracy_closed,
    },
])
leaderboard.to_csv("leaderboard.csv", index=False)
leaderboard
```

</details>

In [ ]:
# Stuck on Exercise 7? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 7" above; your own cell stays as it is.)
get_ipython().set_next_input("def tracker_energy_kwh(tracker_obj):\n    try:\n        return float(tracker_obj.final_emissions_data.energy_consumed)\n    except (AttributeError, TypeError):\n        return None\n\n\nn_params_billion = round(sum(p.numel() for p in model.parameters()) / 1e9, 2)\n\nleaderboard = pd.DataFrame([\n    {\n        \"config\": \"baseline-fp16\",\n        \"model\": MODEL_NAME,\n        \"params_billion\": n_params_billion,\n        \"n_questions\": len(QA_PAIRS),\n        \"duration_sec\": round(elapsed, 2),\n        \"energy_kwh\": tracker_energy_kwh(tracker),\n        \"emissions_kg_co2eq\": emissions_kg,\n        \"accuracy\": accuracy,\n    },\n    {\n        \"config\": \"closed-book-fp16\",\n        \"model\": MODEL_NAME,\n        \"params_billion\": n_params_billion,\n        \"n_questions\": len(QA_PAIRS),\n        \"duration_sec\": round(elapsed_closed, 2),\n        \"energy_kwh\": tracker_energy_kwh(tracker_closed),\n        \"emissions_kg_co2eq\": emissions_closed_kg,\n        \"accuracy\": accuracy_closed,\n    },\n])\nleaderboard.to_csv(\"leaderboard.csv\", index=False)\nleaderboard")


### Where this row fits

In [ ]:
import matplotlib.pyplot as plt

sessions = ["Session 1\n(baseline, today)", "Session 2\n(quantized)", "Session 3\n(+ retrieval)"]
today_value = emissions_kg * 1000
heights = [today_value, today_value, today_value]
colors = ["tab:blue", "lightgray", "lightgray"]
labels = [f"{today_value:.4f} g CO2eq\nmeasured today",
          "measured\nnext session",
          "measured\nin Session 3"]

fig, ax = plt.subplots(figsize=(6.5, 3.5))
bars = ax.bar(sessions, heights, color=colors)
bars[1].set_hatch("//")
bars[2].set_hatch("//")
for bar, label in zip(bars, labels):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.03, label,
            ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, today_value * 1.4)
ax.get_yaxis().set_visible(False)
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.set_title("baseline-fp16: the row later sessions compare against")
plt.tight_layout()
plt.show()

In [ ]:
import os

if os.path.exists("leaderboard.csv"):
    print("Saved leaderboard.csv in this Colab session.")
    print("To download it: click the folder icon in the left sidebar, find")
    print("leaderboard.csv, right-click it, and choose Download.")
else:
    print("leaderboard.csv not found — re-run the cell above first.")

**Key takeaways**

- Handing the model the documents is what makes the quiz answerable: the closed-book
  run shows how little of it the model knows on its own, and how confidently it can
  be wrong.
- `leaderboard.csv` is the empirical baseline later comparisons build on — the chart back
  in Step 7 shows where new rows slot in. Same questions, same scoring, one change at a
  time, using the numbers you personally measured today.

## Step 8 — Share-out (discussion)

Before we wrap up, compare notes with your neighbor.

- What energy and emissions numbers did you get? How close were they to each other's, given
  you likely used the same model on similar Colab hardware?
- What was your accuracy? If it wasn't 100%, look at the wrong answers — what went wrong?
- How many questions did the model answer with no documents? When it was wrong, did it
  say so, or did it give a confident answer anyway?
- Did CodeCarbon report any warnings about not recognizing your hardware? What does that
  tell you about trusting a single carbon number at face value?

## What you learned: check yourself

Answer each question from what you ran and read above. If one is hard, go back to the
step it comes from.

1. What does CodeCarbon measure, and what does it only estimate? Name one reason why two students running
   the same notebook get different numbers.
2. An output says "roughly 1.5 percent" and the reference is `1.5`. Does `score_answer` count it as correct?
   Why does it match whole words only?
3. Which questions could the model answer without the documents? What did it do with the others?
4. With the documents, every input is over a thousand tokens (pieces of words) long; without them, a few dozen.
   Why did the emissions not fall by the same ratio?

## Step 9 — Code literacy check

Before you're done, pick at least two or three code cells above that you actually ran and
read today — from any part of this notebook — and add `#` comments to them explaining, in your
own words, what each meaningful line or block does and *why* it's there, in language a
classmate who hasn't seen the code could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.